In [1]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
    "org.apache.iceberg:iceberg-aws-bundle:1.10.0 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("bronze_taxi")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "password123")
    .getOrCreate()
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.taxi")

taxi_path = "/home/jovyan/project/data/yellow_tripdata_2025-*.parquet"

bronze_df = (
    spark.read.parquet(taxi_path)
    .withColumn("trip_id", F.monotonically_increasing_id())
    .withColumn("source_file", F.input_file_name())
    .withColumn("ingested_at", F.current_timestamp())
)

bronze_df.writeTo("lakehouse.taxi.bronze_trips").createOrReplace()

spark.sql("SELECT COUNT(*) AS bronze_taxi_count FROM lakehouse.taxi.bronze_trips").show()
spark.sql("""
SELECT trip_id, tpep_pickup_datetime, PULocationID, DOLocationID, fare_amount, trip_distance
FROM lakehouse.taxi.bronze_trips
LIMIT 10
""").show(truncate=False)

+-----------------+
|bronze_taxi_count|
+-----------------+
|          7052769|
+-----------------+

+-------+--------------------+------------+------------+-----------+-------------+
|trip_id|tpep_pickup_datetime|PULocationID|DOLocationID|fare_amount|trip_distance|
+-------+--------------------+------------+------------+-----------+-------------+
|0      |2025-01-01 00:18:38 |229         |237         |10.0       |1.6          |
|1      |2025-01-01 00:32:40 |236         |237         |5.1        |0.5          |
|2      |2025-01-01 00:44:04 |141         |141         |5.1        |0.6          |
|3      |2025-01-01 00:14:27 |244         |244         |7.2        |0.52         |
|4      |2025-01-01 00:21:34 |244         |116         |5.8        |0.66         |
|5      |2025-01-01 00:48:24 |239         |68          |19.1       |2.63         |
|6      |2025-01-01 00:14:47 |170         |170         |4.4        |0.4          |
|7      |2025-01-01 00:39:27 |234         |148         |12.1       |1

In [2]:
spark.sql("SHOW TABLES IN lakehouse.taxi").show(truncate=False)

+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|taxi     |bronze_trips|false      |
+---------+------------+-----------+

